In [1]:
%pip install xgboost
import sys
sys.path.append("..")

import pandas as pd
from pathlib import Path
from src.clean import clean_matches, clean_deliveries
from src.features import (
    build_batting_features,
    build_bowling_features,
    build_match_summary,
    build_team_season_stats,
)
RAW = Path(r"E:\ipl-analytics\data\raw")
PROCESSED = Path(r"E:\ipl-analytics\data\processed")


[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [2]:
matches_raw = pd.read_csv(RAW / "matches.csv")
deliveries_raw = pd.read_csv(RAW / "deliveries.csv", low_memory=False)

print(f"RAW matches: {matches_raw.shape}")
print(f"RAW deliveries: {matches_raw.shape}")
print(f"\nRaw season (before cleaning):")
print(sorted(matches_raw["season"].astype(str).unique()))

RAW matches: (1243, 20)
RAW deliveries: (1243, 20)

Raw season (before cleaning):
['2007/08', '2009', '2009/10', '2011', '2012', '2013', '2014', '2015', '2016', '2017', '2018', '2019', '2020/21', '2021', '2022', '2023', '2024', '2025', '2026']


In [3]:
matches = clean_matches(matches_raw)
deliveries = clean_deliveries(deliveries_raw)

OK — all seasons are valid IPL years.
OK — all expected IPL seasons are present.

Season distribution : [np.int32(2008), np.int32(2009), np.int32(2010), np.int32(2011), np.int32(2012), np.int32(2013), np.int32(2014), np.int32(2015), np.int32(2016), np.int32(2017), np.int32(2018), np.int32(2019), np.int32(2020), np.int32(2021), np.int32(2022), np.int32(2023), np.int32(2024), np.int32(2025), np.int32(2026)]
Season range        : 2008 – 2026
Total seasons       : 19
Total matches       : 1243

clean_matches done: (1243, 20) → (1243, 24)
Remaining nulls:
city              51
match_type       148
target_overs     151
super_over       148
method          1222
umpire1          148
umpire2          148

clean_deliveries: (295732, 17) → (295732, 24)
  Nulls remaining:
extras_type    281607
fielder        285011


In [4]:
assert pd.api.types.is_datetime64_any_dtype(matches["date"]), \
   "date column should be datetime64"

matches["season"] = pd.to_numeric(matches["season"], errors='coerce').astype('int64')

assert matches["season"].dtype == int or matches["season"].dtype == "int64", \
    "season should be int"

assert matches["season"].min() == 2008, \
    f"Earliest season should be 2008, got {matches['season'].min()}"

assert 2007 not in matches["season"].values, \
    "2007 should not exist - IPL started in 2008"

assert deliveries["is_wicket"].sum() > 0, \
    "is_ wicket column looks wrong - no wickets found"

assert deliveries["is_four"].sum() > 0, \
    "is_four columns wrong - no fours found"

assert (
    deliveries["match_id"].isin(matches["match_id"]).all()
), "some delivery match_ids have no corresponding match row"   

print("All assertions passed.")

All assertions passed.


In [5]:
print("Raw season values that might be 2020:")
raw_seasons = matches_raw["season"].astype(str).unique()
candidates  = [s for s in raw_seasons if "20" in s]
print(sorted(candidates))

if "date" in matches_raw.columns:
    matches_raw["_date"] = pd.to_datetime(matches_raw["date"], errors="coerce")
    year_2020 = matches_raw[matches_raw["_date"].dt.year == 2020]
    print(f"\nMatches with dates in 2020: {len(year_2020)}")
    if len(year_2020) > 0:
        print(f"Their season values: {year_2020['season'].unique()}")

Raw season values that might be 2020:
['2007/08', '2009', '2009/10', '2011', '2012', '2013', '2014', '2015', '2016', '2017', '2018', '2019', '2020/21', '2021', '2022', '2023', '2024', '2025', '2026']

Matches with dates in 2020: 60
Their season values: <ArrowStringArray>
['2020/21']
Length: 1, dtype: str


In [6]:
deliveries = deliveries.rename(columns={"batter": "batsman"})
batting_stats = build_batting_features(deliveries)
bowling_stats = build_bowling_features(deliveries)
match_summary = build_match_summary(matches, deliveries)
team_season_stats = build_team_season_stats(match_summary)

build_batting_features: 471 batsmans with 30+ balls faced
build_bowling_features: 408 bowlers with 60+ balls bowled
build_match_summary: (1243, 45)
build_team_season_stats: (166, 7)


In [7]:
matches.to_csv(PROCESSED / "matches_clean.csv", index=False)
deliveries.to_csv(PROCESSED / "deliveries_clean.csv", index=False)
batting_stats.to_csv(PROCESSED / "batting_features.csv", index=False)
bowling_stats.to_csv(PROCESSED / "bowling_features.csv", index=False)
match_summary.to_csv(PROCESSED / "match_summary.csv", index=False)
team_season_stats.to_csv(PROCESSED / "team_season_stats.csv", index=False)

print("\nAll files saved to data/processed/")


All files saved to data/processed/


In [8]:
import sqlite3

conn = sqlite3.connect(PROCESSED / "ipl.db")
matches.to_sql("matches", conn, if_exists="replace", index=False)
deliveries.to_sql("deliveries", conn, if_exists="replace", index=False)
batting_stats.to_sql("batting", conn, if_exists="replace", index=False)
bowling_stats.to_sql("bowling", conn, if_exists="replace", index=False)
match_summary.to_sql("match_summary", conn, if_exists="replace", index=False)
team_season_stats.to_sql("team_season", conn, if_exists="replace", index=False)
conn.close()

print("SQLite DB saved: data/processed/ipl.db")

SQLite DB saved: data/processed/ipl.db


In [9]:
print("\n=== OUTPUT SUMMARY ===")
for name, df in {
    "matches_clean" :  matches,
    "deliveries_clean": deliveries,
    "batting_features": batting_stats,
    "bowling_features": bowling_stats,
    "match_summary": match_summary,
    "team_season_stats": team_season_stats,
}.items():
    nulls = df.isnull().sum().sum()
    print(f" {name:<22}: shape={str(df.shape):<16} nulls={nulls}")


=== OUTPUT SUMMARY ===
 matches_clean         : shape=(1243, 24)       nulls=2016
 deliveries_clean      : shape=(295732, 24)     nulls=566618
 batting_features      : shape=(471, 19)        nulls=169
 bowling_features      : shape=(408, 20)        nulls=40
 match_summary         : shape=(1243, 45)       nulls=2070
 team_season_stats     : shape=(166, 7)         nulls=0


In [11]:
print("\n=== Top 10 run scorers ===")
print(batting_stats[["batsman", "total_runs", "batting_average", "strike_rate", "hundreds", "fifties"]].head(10).to_string(index=False))

print("\n=== Top 10 wicket takers ===")
print(bowling_stats[["bowler", "wickets", "economy_rate", "bowling_average", "dot_ball_pct"]].head(10).to_string(index=False))

print("\n=== New columns in deliveries ===")
new_cols = ["is_wicket", "is_four", "is_six", "is_dot_ball", "over_phase", "is_legal_delivery"]
print(deliveries[new_cols].head(10).to_string(index=False))


=== Top 10 run scorers ===
       batsman  total_runs  batting_average  strike_rate  hundreds  fifties
       V Kohli        9346            38.46       134.63         9       68
     RG Sharma        7331            28.86       132.93         2       49
      S Dhawan        6769            35.07       127.62         2       51
     DA Warner        6567            40.04       140.26         4       62
      KL Rahul        5828            45.18       138.83         6       45
      SK Raina        5536            33.15       137.54         1       39
      MS Dhoni        5439            34.42       137.91         0       24
     AM Rahane        5367            29.98       125.10         2       35
     SV Samson        5181            31.98       141.29         5       27
AB de Villiers        5181            41.45       152.38         3       40

=== Top 10 wicket takers ===
     bowler  wickets  economy_rate  bowling_average  dot_ball_pct
    B Kumar      243          7.85      